# ***Corrective RAG***

![image](image.png)

## **Prepare the Vector DB**

In [1]:
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader

In [2]:
loader = PyPDFLoader("data/attentaion-is-all-you-need.pdf")

In [3]:
data = loader.load()

In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size = 400,
    chunk_overlap = 50
)

chunks_docs = splitter.split_documents(data)
len(chunks_docs)

119

## **Load the embedding Model**

In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore

index = faiss.IndexFlatIP(len(embeddings.embed_query("hello")))

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={}
)

In [7]:
from uuid import uuid4

uuids = [str(uuid4()) for _ in range(len(chunks_docs))]

In [8]:
vector_store.add_documents(documents=chunks_docs, ids=uuids)
retriever = vector_store.as_retriever(
    search_kwargs = {
        "k": 6
    }
)

## **Test Query**

In [9]:
retriever.invoke(input="What is autoregressive?")

[Document(id='2f5e4759-e7af-4ddc-a6cc-861d69cf0b73', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/attentaion-is-all-you-need.pdf', 'total_pages': 15, 'page': 1, 'page_label': '2'}, page_content='sequence (y1, ..., ym) of symbols one element at a time. At each step the model is auto-regressive\n[10], consuming the previously generated symbols as additional input when generating the next.\n2'),
 Document(id='168fd3af-cb8e-4929-84da-5abbc63d5a57', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 